# MVP de Engenharia de Dados
## Notebook 5 — Camada Gold

### Objetivo deste notebook

Este notebook é responsável pela construção da camada Gold do pipeline de dados.

A camada Gold utilizará como fonte os dados tratados e validados disponíveis na camada Silver:

`workspace.energia_silver.balanco_energia_subsistema`

Nesta etapa, os dados serão modelados e agregados para facilitar o consumo analítico e permitir que as perguntas de negócio definidas no início do MVP sejam respondidas de forma estruturada.

### Estratégia de modelagem

A modelagem da camada Gold será orientada pelas análises de:

- evolução da carga de energia ao longo do tempo e entre os subsistemas;
- distribuição da geração entre as fontes hidráulica, térmica, eólica e solar;
- evolução da participação das diferentes fontes de geração;
- relação entre geração e carga ao longo do período analisado.

Serão construídas estruturas analíticas derivadas da camada Silver, preservando a rastreabilidade entre os dados tratados e os resultados utilizados nas análises.

### Entrada

Tabela Delta da camada Silver:

`workspace.energia_silver.balanco_energia_subsistema`

### Saída esperada

Tabelas analíticas persistidas no schema:

`workspace.energia_gold`


In [0]:
tabela_silver = "workspace.energia_silver.balanco_energia_subsistema"

df_silver = spark.table(tabela_silver)

print("Tabela Silver carregada com sucesso.")
print("Quantidade de registros:", df_silver.count())
print("Quantidade de colunas:", len(df_silver.columns))

Tabela Silver carregada com sucesso.
Quantidade de registros: 263040
Quantidade de colunas: 13


In [0]:
from pyspark.sql.functions import avg, min, max, round

df_gold_carga_mensal = (
    df_silver
    .groupBy(
        "ano",
        "mes",
        "id_subsistema",
        "nom_subsistema"
    )
    .agg(
        round(avg("val_carga"), 2).alias("carga_media"),
        round(min("val_carga"), 2).alias("carga_minima"),
        round(max("val_carga"), 2).alias("carga_maxima")
    )
    .orderBy(
        "ano",
        "mes",
        "id_subsistema"
    )
)

display(df_gold_carga_mensal.limit(20))

ano,mes,id_subsistema,nom_subsistema,carga_media,carga_minima,carga_maxima
2020,1,N,NORTE,5412.0,4260.09,6101.37
2020,1,NE,NORDESTE,10952.56,8203.38,12787.02
2020,1,S,SUL,12599.13,7791.72,17403.36
2020,1,SE,SUDESTE/CENTRO-OESTE,39577.57,27453.61,51230.32
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,68541.26,49362.19,84879.02
2020,2,N,NORTE,5513.53,4523.39,6355.08
2020,2,NE,NORDESTE,10995.71,8853.08,12983.98
2020,2,S,SUL,12781.99,7703.28,18002.31
2020,2,SE,SUDESTE/CENTRO-OESTE,39505.45,27405.48,50626.06
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,68796.67,49256.81,86509.74


In [0]:
quantidade_gold_carga = df_gold_carga_mensal.count()

print("Quantidade de registros na Gold de carga mensal:", quantidade_gold_carga)

Quantidade de registros na Gold de carga mensal: 360


In [0]:
tabela_gold_carga = "workspace.energia_gold.carga_mensal_subsistema"

df_gold_carga_mensal.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabela_gold_carga)

print("Tabela Gold de carga mensal criada com sucesso:")
print(tabela_gold_carga)

Tabela Gold de carga mensal criada com sucesso:
workspace.energia_gold.carga_mensal_subsistema


In [0]:
from pyspark.sql.functions import avg, round

df_gold_geracao_mensal = (
    df_silver
    .groupBy(
        "ano",
        "mes",
        "id_subsistema",
        "nom_subsistema"
    )
    .agg(
        round(avg("val_gerhidraulica"), 2).alias("geracao_hidraulica_media"),
        round(avg("val_gertermica"), 2).alias("geracao_termica_media"),
        round(avg("val_gereolica"), 2).alias("geracao_eolica_media"),
        round(avg("val_gersolar"), 2).alias("geracao_solar_media")
    )
    .orderBy(
        "ano",
        "mes",
        "id_subsistema"
    )
)

display(df_gold_geracao_mensal.limit(20))

ano,mes,id_subsistema,nom_subsistema,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media
2020,1,N,NORTE,8188.34,2861.9,126.65,0.0
2020,1,NE,NORDESTE,2897.49,2322.59,2965.7,317.59
2020,1,S,SUL,4523.55,1817.24,517.36,0.0
2020,1,SE,SUDESTE/CENTRO-OESTE,35469.7,6309.17,0.0,192.27
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,51079.08,13310.89,3609.71,509.87
2020,2,N,NORTE,12613.98,2140.81,99.05,0.0
2020,2,NE,NORDESTE,2883.37,1697.99,3086.87,333.01
2020,2,S,SUL,3361.24,1087.89,708.2,0.0
2020,2,SE,SUDESTE/CENTRO-OESTE,36109.29,4494.53,0.0,180.45
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,54967.88,9421.22,3894.12,513.46


In [0]:
quantidade_gold_geracao = df_gold_geracao_mensal.count()

print(
    "Quantidade de registros na Gold de geração mensal:",
    quantidade_gold_geracao
)

Quantidade de registros na Gold de geração mensal: 360


In [0]:
tabela_gold_geracao = "workspace.energia_gold.geracao_mensal_subsistema"

df_gold_geracao_mensal.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabela_gold_geracao)

print("Tabela Gold de geração mensal criada com sucesso:")
print(tabela_gold_geracao)

Tabela Gold de geração mensal criada com sucesso:
workspace.energia_gold.geracao_mensal_subsistema


In [0]:
from pyspark.sql.functions import col, round, when

df_gold_participacao = (
    df_gold_geracao_mensal
    .withColumn(
        "geracao_total_media",
        col("geracao_hidraulica_media") +
        col("geracao_termica_media") +
        col("geracao_eolica_media") +
        col("geracao_solar_media")
    )
    .withColumn(
        "participacao_hidraulica_pct",
        when(
            col("geracao_total_media") != 0,
            round(
                col("geracao_hidraulica_media") /
                col("geracao_total_media") * 100, 2
            )
        )
    )
    .withColumn(
        "participacao_termica_pct",
        when(
            col("geracao_total_media") != 0,
            round(
                col("geracao_termica_media") /
                col("geracao_total_media") * 100, 2
            )
        )
    )
    .withColumn(
        "participacao_eolica_pct",
        when(
            col("geracao_total_media") != 0,
            round(
                col("geracao_eolica_media") /
                col("geracao_total_media") * 100, 2
            )
        )
    )
    .withColumn(
        "participacao_solar_pct",
        when(
            col("geracao_total_media") != 0,
            round(
                col("geracao_solar_media") /
                col("geracao_total_media") * 100, 2
            )
        )
    )
)

display(df_gold_participacao.limit(20))

ano,mes,id_subsistema,nom_subsistema,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media,geracao_total_media,participacao_hidraulica_pct,participacao_termica_pct,participacao_eolica_pct,participacao_solar_pct
2020,1,N,NORTE,8188.34,2861.9,126.65,0.0,11176.89,73.26,25.61,1.13,0.0
2020,1,NE,NORDESTE,2897.49,2322.59,2965.7,317.59,8503.369999999999,34.07,27.31,34.88,3.73
2020,1,S,SUL,4523.55,1817.24,517.36,0.0,6858.15,65.96,26.5,7.54,0.0
2020,1,SE,SUDESTE/CENTRO-OESTE,35469.7,6309.17,0.0,192.27,41971.13999999999,84.51,15.03,0.0,0.46
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,51079.08,13310.89,3609.71,509.87,68509.55,74.56,19.43,5.27,0.74
2020,2,N,NORTE,12613.98,2140.81,99.05,0.0,14853.839999999998,84.92,14.41,0.67,0.0
2020,2,NE,NORDESTE,2883.37,1697.99,3086.87,333.01,8001.24,36.04,21.22,38.58,4.16
2020,2,S,SUL,3361.24,1087.89,708.2,0.0,5157.33,65.17,21.09,13.73,0.0
2020,2,SE,SUDESTE/CENTRO-OESTE,36109.29,4494.53,0.0,180.45,40784.27,88.54,11.02,0.0,0.44
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,54967.88,9421.22,3894.12,513.46,68796.68000000001,79.9,13.69,5.66,0.75


In [0]:
from pyspark.sql.functions import col, round, min, max

df_validacao_percentual = (
    df_gold_participacao
    .withColumn(
        "soma_participacoes",
        round(
            col("participacao_hidraulica_pct") +
            col("participacao_termica_pct") +
            col("participacao_eolica_pct") +
            col("participacao_solar_pct"),
            2
        )
    )
)

df_validacao_percentual.select(
    min("soma_participacoes").alias("menor_soma"),
    max("soma_participacoes").alias("maior_soma")
).show()

+----------+----------+
|menor_soma|maior_soma|
+----------+----------+
|     99.99|    100.01|
+----------+----------+



In [0]:
tabela_gold_participacao = "workspace.energia_gold.participacao_fontes_mensal"

df_gold_participacao.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabela_gold_participacao)

print("Tabela Gold de participação das fontes criada com sucesso:")
print(tabela_gold_participacao)

Tabela Gold de participação das fontes criada com sucesso:
workspace.energia_gold.participacao_fontes_mensal


In [0]:
from pyspark.sql.functions import col, round

df_gold_geracao_carga = (
    df_gold_carga_mensal.alias("carga")
    .join(
        df_gold_geracao_mensal.alias("geracao"),
        on=[
            "ano",
            "mes",
            "id_subsistema",
            "nom_subsistema"
        ],
        how="inner"
    )
    .withColumn(
        "geracao_total_media",
        round(
            col("geracao_hidraulica_media") +
            col("geracao_termica_media") +
            col("geracao_eolica_media") +
            col("geracao_solar_media"),
            2
        )
    )
    .select(
        "ano",
        "mes",
        "id_subsistema",
        "nom_subsistema",
        "carga_media",
        "geracao_hidraulica_media",
        "geracao_termica_media",
        "geracao_eolica_media",
        "geracao_solar_media",
        "geracao_total_media"
    )
    .orderBy(
        "ano",
        "mes",
        "id_subsistema"
    )
)

display(df_gold_geracao_carga.limit(20))

ano,mes,id_subsistema,nom_subsistema,carga_media,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media,geracao_total_media
2020,1,N,NORTE,5412.0,8188.34,2861.9,126.65,0.0,11176.89
2020,1,NE,NORDESTE,10952.56,2897.49,2322.59,2965.7,317.59,8503.37
2020,1,S,SUL,12599.13,4523.55,1817.24,517.36,0.0,6858.15
2020,1,SE,SUDESTE/CENTRO-OESTE,39577.57,35469.7,6309.17,0.0,192.27,41971.14
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,68541.26,51079.08,13310.89,3609.71,509.87,68509.55
2020,2,N,NORTE,5513.53,12613.98,2140.81,99.05,0.0,14853.84
2020,2,NE,NORDESTE,10995.71,2883.37,1697.99,3086.87,333.01,8001.24
2020,2,S,SUL,12781.99,3361.24,1087.89,708.2,0.0,5157.33
2020,2,SE,SUDESTE/CENTRO-OESTE,39505.45,36109.29,4494.53,0.0,180.45,40784.27
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,68796.67,54967.88,9421.22,3894.12,513.46,68796.68


In [0]:
quantidade_gold_geracao_carga = df_gold_geracao_carga.count()

print(
    "Quantidade de registros na Gold de geração e carga:",
    quantidade_gold_geracao_carga
)

Quantidade de registros na Gold de geração e carga: 360


In [0]:
from pyspark.sql.functions import avg, min, max, round

df_silver \
    .groupBy("id_subsistema", "nom_subsistema") \
    .agg(
        round(avg("val_intercambio"), 2).alias("intercambio_medio"),
        round(min("val_intercambio"), 2).alias("intercambio_minimo"),
        round(max("val_intercambio"), 2).alias("intercambio_maximo")
    ) \
    .orderBy("id_subsistema") \
    .show(truncate=False)

+-------------+----------------------------+-----------------+------------------+------------------+
|id_subsistema|nom_subsistema              |intercambio_medio|intercambio_minimo|intercambio_maximo|
+-------------+----------------------------+-----------------+------------------+------------------+
|N            |NORTE                       |3430.21          |-4535.01          |14146.51          |
|NE           |NORDESTE                    |3886.69          |-7872.62          |13591.51          |
|S            |SUL                         |-2171.73         |-11215.26         |8425.68           |
|SE           |SUDESTE/CENTRO-OESTE        |-4992.37         |-16172.03         |7167.82           |
|SIN          |SISTEMA INTERLIGADO NACIONAL|152.2            |-2766.58          |2700.75           |
+-------------+----------------------------+-----------------+------------------+------------------+



In [0]:
from pyspark.sql.functions import avg, round

df_intercambio_mensal = (
    df_silver
    .groupBy(
        "ano",
        "mes",
        "id_subsistema",
        "nom_subsistema"
    )
    .agg(
        round(
            avg("val_intercambio"), 2
        ).alias("intercambio_liquido_medio")
    )
)

print(
    "Quantidade de registros do intercâmbio mensal:",
    df_intercambio_mensal.count()
)

display(df_intercambio_mensal.limit(20))

Quantidade de registros do intercâmbio mensal: 360


ano,mes,id_subsistema,nom_subsistema,intercambio_liquido_medio
2020,1,NE,NORDESTE,-2449.19
2020,1,N,NORTE,5764.89
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,-31.72
2020,1,SE,SUDESTE/CENTRO-OESTE,2393.58
2020,1,S,SUL,-5740.99
2020,2,NE,NORDESTE,-2994.47
2020,2,N,NORTE,9340.31
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,-0.01
2020,2,SE,SUDESTE/CENTRO-OESTE,1278.81
2020,2,S,SUL,-7624.67


In [0]:
df_gold_geracao_carga = (
    df_gold_geracao_carga
    .join(
        df_intercambio_mensal,
        on=[
            "ano",
            "mes",
            "id_subsistema",
            "nom_subsistema"
        ],
        how="inner"
    )
    .select(
        "ano",
        "mes",
        "id_subsistema",
        "nom_subsistema",
        "carga_media",
        "geracao_hidraulica_media",
        "geracao_termica_media",
        "geracao_eolica_media",
        "geracao_solar_media",
        "geracao_total_media",
        "intercambio_liquido_medio"
    )
    .orderBy(
        "ano",
        "mes",
        "id_subsistema"
    )
)

print(
    "Quantidade de registros após adicionar o intercâmbio:",
    df_gold_geracao_carga.count()
)

display(df_gold_geracao_carga.limit(20))

Quantidade de registros após adicionar o intercâmbio: 360


ano,mes,id_subsistema,nom_subsistema,carga_media,geracao_hidraulica_media,geracao_termica_media,geracao_eolica_media,geracao_solar_media,geracao_total_media,intercambio_liquido_medio
2020,1,N,NORTE,5412.0,8188.34,2861.9,126.65,0.0,11176.89,5764.89
2020,1,NE,NORDESTE,10952.56,2897.49,2322.59,2965.7,317.59,8503.37,-2449.19
2020,1,S,SUL,12599.13,4523.55,1817.24,517.36,0.0,6858.15,-5740.99
2020,1,SE,SUDESTE/CENTRO-OESTE,39577.57,35469.7,6309.17,0.0,192.27,41971.14,2393.58
2020,1,SIN,SISTEMA INTERLIGADO NACIONAL,68541.26,51079.08,13310.89,3609.71,509.87,68509.55,-31.72
2020,2,N,NORTE,5513.53,12613.98,2140.81,99.05,0.0,14853.84,9340.31
2020,2,NE,NORDESTE,10995.71,2883.37,1697.99,3086.87,333.01,8001.24,-2994.47
2020,2,S,SUL,12781.99,3361.24,1087.89,708.2,0.0,5157.33,-7624.67
2020,2,SE,SUDESTE/CENTRO-OESTE,39505.45,36109.29,4494.53,0.0,180.45,40784.27,1278.81
2020,2,SIN,SISTEMA INTERLIGADO NACIONAL,68796.67,54967.88,9421.22,3894.12,513.46,68796.68,-0.01


In [0]:
tabela_gold_geracao_carga = "workspace.energia_gold.geracao_carga_mensal"

df_gold_geracao_carga.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_gold_geracao_carga)

print("Tabela Gold atualizada com sucesso:")
print(tabela_gold_geracao_carga)

Tabela Gold atualizada com sucesso:
workspace.energia_gold.geracao_carga_mensal


In [0]:
%sql

SELECT 'carga_mensal_subsistema' AS tabela, COUNT(*) AS registros
FROM workspace.energia_gold.carga_mensal_subsistema

UNION ALL

SELECT 'geracao_mensal_subsistema', COUNT(*)
FROM workspace.energia_gold.geracao_mensal_subsistema

UNION ALL

SELECT 'participacao_fontes_mensal', COUNT(*)
FROM workspace.energia_gold.participacao_fontes_mensal

UNION ALL

SELECT 'geracao_carga_mensal', COUNT(*)
FROM workspace.energia_gold.geracao_carga_mensal;


tabela,registros
carga_mensal_subsistema,360
geracao_mensal_subsistema,360
participacao_fontes_mensal,360
geracao_carga_mensal,360


## Resultado da Camada Gold

A camada Gold foi construída a partir dos dados tratados e padronizados disponíveis na camada Silver, com o objetivo de disponibilizar estruturas analíticas diretamente relacionadas às perguntas de negócio definidas para o MVP.

Para as análises temporais, foi adotada a granularidade mensal por identificador de subsistema. Dessa forma, cada registro representa a combinação de um mês com um identificador de subsistema.

Foram utilizadas médias mensais para as variáveis de carga e geração, preservando a característica dos valores horários presentes na fonte e evitando interpretar uma soma simples desses registros como uma medida de energia acumulada.

### Tabelas criadas

**1. `workspace.energia_gold.carga_mensal_subsistema`**

Contém indicadores mensais de carga por identificador de subsistema:
- carga média;
- carga mínima;
- carga máxima.

Essa estrutura será utilizada principalmente para responder à primeira pergunta de negócio, relacionada à variação da carga ao longo do tempo e entre os subsistemas.

---

**2. `workspace.energia_gold.geracao_mensal_subsistema`**

Contém as médias mensais das fontes de geração disponíveis no conjunto de dados:
- geração hidráulica;
- geração térmica;
- geração eólica;
- geração solar.

Essa estrutura permite analisar a distribuição e o comportamento temporal das diferentes fontes de geração.

---

**3. `workspace.energia_gold.participacao_fontes_mensal`**

Apresenta a participação percentual mensal das quatro fontes de geração consideradas no projeto.

A geração total utilizada para o cálculo corresponde à soma das gerações hidráulica, térmica, eólica e solar presentes no conjunto de dados. Portanto, as participações calculadas representam especificamente essas quatro fontes consideradas na análise.

Na validação, a soma das participações apresentou valores entre 99,99% e 100,01%, diferença decorrente do arredondamento para duas casas decimais.

---

**4. `workspace.energia_gold.geracao_carga_mensal`**

Reúne, na mesma granularidade mensal:
- carga média;
- geração hidráulica média;
- geração térmica média;
- geração eólica média;
- geração solar média;
- geração total média das quatro fontes consideradas;
- intercâmbio líquido médio.

A inclusão do intercâmbio permite complementar a análise da relação entre geração e carga nos diferentes subsistemas. Conforme a documentação da fonte, o intercâmbio representa o valor verificado do intercâmbio líquido e pode assumir valores negativos. Neste projeto, não será atribuído um sentido específico ao sinal positivo ou negativo sem documentação que estabeleça explicitamente essa convenção.

Essa estrutura será utilizada principalmente para responder à quarta pergunta de negócio.

---

### Validação da camada Gold

As quatro tabelas foram persistidas no formato Delta no schema `workspace.energia_gold`.

Cada tabela possui **360 registros**, correspondentes a:

**6 anos × 12 meses × 5 identificadores = 360 registros**

A validação final confirmou a persistência das quatro estruturas no catálogo do Databricks.

Com a camada Gold concluída, os dados encontram-se organizados e preparados para a etapa de análise e resposta às perguntas de negócio.
